# 🎭 SadTalker - AI Talking Head Generator
## สำหรับ TikTok Live AI Avatar

**ขั้นตอน:
1. Run ขั้นที่ 1 - ติดตั้ง SadTalker
2. Run ขั้นที่ 2 - Upload รูป Avatar ของคุณ
3. Run ขั้นที่ 3 - Upload เสียง TTS หรือ พิมพ์ข้อความ
4. Run ขั้นที่ 4 - สร้าง Talking Head Video
5. Run ขั้นที่ 5 - ดาวน์โหลดวิดีโอ

In [ ]:
# 📦 ขั้นที่ 1: ติดตั้ง SadTalker
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -r requirements.txt
!pip install onnxruntime-gpu
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%cd ..

In [ ]:
# 📤 ขั้นที่ 2: Upload รูป Avatar
# รัน cell นี้แล้วจะมีปุ่มให้ upload รูป
from google.colab import files
import os

# สร้างโฟลเดอร์
!mkdir -p SadTalker/input

print("กดปุ่ม Upload เพื่อเลือกรูป Avatar ของคุณ (.jpg, .png)")
uploaded = files.upload()

# บันทึกรูป
for filename in uploaded.keys():
    img_path = f"SadTalker/input/{filename}"
    with open(img_path, 'wb') as f:
        f.write(uploaded[filename])
    print(f"✅ บันทึกรูปที่: {img_path}")

In [ ]:
# 🔊 ขั้นที่ 3: สร้างเสียง TTS (เลือกวิธีที่ต้องการ)

# === วิธีที่ 1: ใช้ Google TTS (ฟรี) ===
!pip install gTTS

from gtts import gTTS
import os

# พิมพ์ข้อความที่อยากให้ AI พูด
text = "สวัสดีครับ ยินดีต้อนรับเข้าสู่ Live ของเราครับ วันนี้ผมจะพาทุกคนดูบ้านที่สวยมากๆเลยนะครับ"

tts = gTTS(text=text, lang='th')
audio_path = "SadTalker/input/demo.mp3"
tts.save(audio_path)
print(f"✅ สร้างเสียงสำเร็จ: {audio_path}")

# === วิธีที่ 2: ใช้ ElevenLabs (ต้องมี API Key) ===
# ถอดcomment ถ้าต้องการใช้ ElevenLabs
# !pip install elevenlabs
# from elevenlabs import generate, set_api_key
# set_api_key("YOUR_ELEVENLABS_API_KEY")
# audio = generate(text="สวัสดีครับ", voice="Thailand Male", model="elevenlabs_th")
# with open("SadTalker/input/demo.mp3", "wb") as f:
#     f.write(audio)

# === วิธีที่ 3: Upload ไฟล์เสียงที่มีอยู่แล้ว ===
# ถอดcomment แล้วuploadไฟล์ .mp3
# uploaded_audio = files.upload()
# for filename in uploaded_audio.keys():
#     !mv "{filename}" SadTalker/input/demo.mp3

In [ ]:
# 🎬 ขั้นที่ 4: สร้าง Talking Head Video
import os
import sys
sys.path.append('SadTalker')

import torch

# เช็ค GPU
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ตั้งค่า paths
 SadTalker_DIR = "SadTalker"
 checkpoints_DIR = os.path.join(SadTalker_DIR, "checkpoints")
 img_path = "SadTalker/input/"
 audio_path = "SadTalker/input/demo.mp3"

# หารูปล่าสุด
import glob
img_files = glob.glob(os.path.join(img_path, "*.jpg")) + glob.glob(os.path.join(img_path, "*.png"))
if img_files:
    img_path = img_files[0]
    print(f"ใช้รูป: {img_path}")
else:
    print("❌ ไม่พบรูป กรุณา upload ก่อน")
    img_path = None

if img_path and os.path.exists(audio_path):
    # สร้าง output path
    output_path = "SadTalker/output/result.mp4"
    !mkdir -p SadTalker/output
    
    # รัน SadTalker
    !python SadTalker/inference.py \
        --driven_audio {audio_path} \
        --source_image {img_path} \
        --result_dir SadTalker/output \
        --enhancer gfpgan \
        --batch_size 1 \
        --input_yaw_mean 0 \
        --expression_scale 1.0 \
        --motion_scale 1.0
    
    print(f"✅ สร้างวิดีโอสำเร็จ: {output_path}")
else:
    print("❌ กรุณาupload รูปและเสียงก่อน")

In [ ]:
# ⬇️ ขั้นที่ 5: ดาวน์โหลดวิดีโอ
from google.colab import files
import os

output_dir = "SadTalker/output"

# หาไฟล์ล่าสุด
video_files = glob.glob(os.path.join(output_dir, "*.mp4"))
if video_files:
    latest_video = max(video_files, key=os.path.getmtime)
    print(f"ดาวน์โหลด: {latest_video}")
    files.download(latest_video)
else:
    print("❌ ไม่พบวิดีโอ")

# แสดงวิดีโอใน Colab
from IPython.display import HTML, Video
if video_files:
    display(Video(latest_video, embed=True))

---

## 📋 วิธีใช้สำหรับ TikTok Live AI

### 1. ใช้ OBS Studio
1. ดาวน์โหลด [OBS Studio](https://obsproject.com)
2. เพิ่ม **Media Source** 
3. ใส่วิดีโอที่ดาวน์โหลด หรือใช้ **Image** source สำหรับรูปนิ่ง
4. ตั้งค่า **Loop**

### 2. เชื่อมกับ AI Chat + TTS
1. สร้าง Web App ที่แสดง Avatar + ข้อความแชท
2. ใช้ OBS จับหน้าจอ Web App
3. เชื่อมกับ MiniMax/GPT-4 สำหรับตอบแชท
4. ใช้ MiniMax TTS หรือ ElevenLabs สร้างเสียง

### 3. ตั้งค่า TikTok Live
1. เปิด TikTok → ไอคอน + → Live
2. ใช้ **RTMP URL** จาก TikTok
3. ตั้งค่า OBS → Settings → Stream
4. ใส่ RTMP URL แล้ว Start Stream

---

**💡 Tips:**
- ใช้วิดีโอสั้นๆ 5-10 วินาที แล้ว loop ซ้ำ
- เตรียมวิดีโอหลายแบบ (ทักทาย, ขายของ, จบ Live)
- ใช้ `ffmpeg` ตัดต่อวิดีโอให้เหมาะกับ TikTok